# **Demo 01: Building a CrewAI-Powered Agent With OpenAI**


**Objective:** To demonstrate how to build a multi-agent system using CrewAI that intelligently identifies queries requiring real-time information and retrieves the most up-to-date data accordingly.

**Prerequisites:** OpenAI key

**Tools required:** Python

**Scenario:** A product manager at a digital knowledge platform needs to manage an increasing volume of user queries, many of which require real-time updates and dynamic information. At present, these queries are reviewed and answered manually, leading to delays, inconsistencies, and rising operational costs. The goal is to automate the query-handling process using a multi-agent system that can identify when real-time data retrieval is needed and provide users with fast, accurate, and context-aware responses while maintaining efficiency and cost control. 

In [1]:
#Step 1: Install the required libraries
#Tavily is a tool that allows us to perform dynamic web searches, which is crucial for our agent to access up-to-date information from the web.
#Pydantic is for schema formatting, which helps us define the structure of our tool's input and output.
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic python-dotenv

Defaulting to user installation because normal site-packages is not writeable
  Using cached openai-2.32.0-py3-none-any.whl.metadata (31 kB)
Using cached openai-2.32.0-py3-none-any.whl (1.2 MB)
  Attempting uninstall: openai
    Found existing installation: openai 2.24.0
    Uninstalling openai-2.24.0:
      Successfully uninstalled openai-2.24.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\AgrawalN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import requests
from dotenv import load_dotenv
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool

# Load API keys from .env file
load_dotenv()

True

In [ ]:
# ---- Custom CrewAI Tool for Web Search and behaves like the base tool----
# Every tool has a run function that defines what happens when the agent uses the tool.
# In this case, it makes a request to the Tavily API with the user's query and returns formatted search results.
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 5
        }

        response = requests.post(url, json=payload)
        data = response.json()

        results = []
        for r in data["results"]:
            results.append(f"{r['title']} - {r['url']}")

        return "\n".join(results)


search_tool = TavilySearchTool()

# ---- Groq LLM (using OpenAI-compatible endpoint) ----
llm = LLM(
    model="llama-3.3-70b-versatile",
    provider="openai",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    temperature=0.5
)

Unable to initialize LLM with model 'openai/llama-3.3-70b-versatile'. The model did not match any supported native provider (openai, anthropic, claude, azure, azure_openai, google, gemini, bedrock, aws, openrouter, deepseek, ollama, ollama_chat, hosted_vllm, cerebras, dashscope), and the LiteLLM fallback package is not installed.

To fix this, either:
  1. Install LiteLLM for broad model support: uv add 'crewai[litellm]'
or
pip install litellm

For more details, see: https://docs.crewai.com/en/learn/llm-connections


ImportError: Unable to initialize LLM with model 'openai/llama-3.3-70b-versatile'. The model did not match any supported native provider (openai, anthropic, claude, azure, azure_openai, google, gemini, bedrock, aws, openrouter, deepseek, ollama, ollama_chat, hosted_vllm, cerebras, dashscope), and the LiteLLM fallback package is not installed.

To fix this, either:
  1. Install LiteLLM for broad model support: uv add 'crewai[litellm]'
or
pip install litellm

For more details, see: https://docs.crewai.com/en/learn/llm-connections

In [ ]:
# Researcher agent
# Only 1 instance of the agent
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for FMCG",
    backstory="You are an expert in artificial intelligence and stay updated with the latest research trends in FMCG.",
    verbose=True, # important to to turn on logging.
    allow_delegation=False, # Hierarchy coordination (no delegation)
    llm=llm, # could customize the llm being used. We could have a unique llm per agent.
    # Set up the tools  which the agent can use.
    tools = [search_tool]
)

# Writer agent
writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory="You are an experienced technical writer with expertise in summarizing research for executives.",
    verbose=True,
    allow_delegation=False, 
    llm=llm,
    tools = [search_tool]
)


In [ ]:
# ---- Tasks ----
# what are tasks? Tasks are like prompts for the agents, but they also include an expected output and can be assigned to specific agents.
task_research = Task(
    description="Search the web and identify the top 3 recent advancements in AI for FMCG.",
    expected_output="Detailed notes explaining three recent AI advancements in FMCG with examples.",
    agent=researcher # Assign an agent to the task (the researcher will perform the research and gather information)
)

# This task will be second so it can use research as context.
task_write = Task(
    description="""
Write a concise executive summary using the research notes.

Requirements:
- Maximum 200 words
- Use bullet points
- Focus only on the 3 key advancements
""",
    expected_output="Executive summary of AI advancements in FMCG.",
    agent=writer, # writer agent
    context=[task_research] #uses the output of the research task as context to write the executive summary
)

In [ ]:
# ---- Crew ----
#Multi agent system and we kick off the crew.
crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write], # in order of tasks.
    verbose=True
)

result = crew.kickoff()

print("\nFinal Output:\n")
print(result.raw)